# Exploratory Data Analysis – Department Performance and Key Factors

**Business questions answered here**
1. How does performance differ by department (and job role)?
2. Which three factors influence employee performance the most?

Methods: distribution checks, statistical tests, and three independent ways of ranking factors (effect size, model importance, permutation importance).

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

ROOT = Path("../..").resolve()
RAW_FILE = ROOT / "data" / "raw" / "INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.xls"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "src" / "models"
FIGURES = ROOT / "src" / "visualization" / "figures"
TARGET = "PerformanceRating"
RANDOM_STATE = 42

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
data = pd.read_csv(PROCESSED / "employee_cleaned.csv")
labelled = pd.read_csv(PROCESSED / "employee_labelled.csv")
data.shape

(1200, 28)

## 1. Performance rating distribution

In [2]:
counts = data[TARGET].value_counts().sort_index()
summary = pd.DataFrame({"employees": counts, "percentage": (counts / len(data) * 100).round(1)})
summary.index.name = "PerformanceRating"
summary

,employees,percentage
PerformanceRating,,
2,194,16.2
3,874,72.8
4,132,11.0


Rating 1 does not occur in the data; ratings are 2 (Good), 3 (Excellent) and 4 (Outstanding). The data is **imbalanced**, so accuracy alone can be misleading and macro averaged scores are used later.

## 2. Department wise performance

In [3]:
by_department = data.groupby("EmpDepartment")[TARGET].agg(employees="count", average_rating="mean").round(3)
share = (pd.crosstab(data["EmpDepartment"], data[TARGET], normalize="index") * 100).round(1)
share.columns = [f"percent_rating_{c}" for c in share.columns]
department_table = by_department.join(share).sort_values("average_rating", ascending=False)
department_table

,employees,average_rating,percent_rating_2,percent_rating_3,percent_rating_4
EmpDepartment,,,,,
Development,361,3.086,3.6,84.2,12.2
Data Science,20,3.050,5.0,85.0,10.0
Human Resources,54,2.926,18.5,70.4,11.1
Research & Development,343,2.921,19.8,68.2,12.0
Sales,373,2.861,23.3,67.3,9.4
Finance,49,2.776,30.6,61.2,8.2


In [4]:
groups = [g[TARGET].values for _, g in data.groupby("EmpDepartment")]
h_statistic, p_kruskal = stats.kruskal(*groups)
contingency = pd.crosstab(data["EmpDepartment"], data[TARGET])
chi2, p_chi, dof, _ = stats.chi2_contingency(contingency)
print(f"Kruskal-Wallis test across departments: statistic = {h_statistic:.2f}, p-value = {p_kruskal:.4f}")
print(f"Chi-square test (department vs rating): statistic = {chi2:.2f}, p-value = {p_chi:.4f}")

Kruskal-Wallis test across departments: statistic = 44.92, p-value = 0.0000
Chi-square test (department vs rating): statistic = 70.33, p-value = 0.0000


**Findings – department wise performance**

| Department | Employees | Average rating | Percent rated 2 (lowest) |
|---|---|---|---|
| Development | 361 | 3.09 | 3.6 |
| Data Science | 20 | 3.05 | 5.0 |
| Human Resources | 54 | 2.93 | 18.5 |
| Research & Development | 343 | 2.92 | 19.8 |
| Sales | 373 | 2.86 | 23.3 |
| Finance | 49 | 2.78 | 30.6 |

- Development and Data Science perform best; **Finance is the weakest** (nearly one in three employees rated 2) followed by **Sales**, which is also the largest group with a low rating.
- The differences between departments are statistically significant (Kruskal-Wallis and chi-square p-values below 0.001).
- Data Science has only 20 employees, so its result should be read with care.

In [5]:
by_role = data.groupby(["EmpDepartment", "EmpJobRole"])[TARGET].agg(employees="count", average_rating="mean").round(3)
by_role.sort_values("average_rating", ascending=False)

employees  average_rating
EmpDepartment          EmpJobRole                                          
Research & Development Manager                            8           3.250
Development            Business Analyst                  16           3.188
                       Developer                        236           3.106
                       Technical Lead                    38           3.079
Data Science           Data Scientist                    20           3.050
Development            Senior Developer                  52           3.000
                       Delivery Manager                  12           3.000
Human Resources        Manager                            9           3.000
Development            Technical Architect                7           3.000
Research & Development Manufacturing Director            33           2.970
                       Research Scientist                77           2.961
                       Manager R&D                       94           2.936
Human Resources        Human Resources                   45           2.911
Research & Development Research Director                 19           2.895
Sales                  Manager                           34           2.882
                       Sales Representative              69           2.870
Research & Development Laboratory Technician             64           2.859
Sales                  Sales Executive                  270           2.856
Research & Development Healthcare Representative         33           2.848
                       Senior Manager R&D                15           2.800
Finance                Finance Manager                   49           2.776

## 3. Which factors relate to performance?

### 3.1 Statistical association with the rating

In [6]:
features = [c for c in data.columns if c not in ("EmpNumber", TARGET)]
categorical = [c for c in features if not pd.api.types.is_numeric_dtype(data[c])]
numeric = [c for c in features if c not in categorical]

def cramers_v(a, b):
    table = pd.crosstab(a, b)
    chi2 = stats.chi2_contingency(table)[0]
    n = table.values.sum()
    r, k = table.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

rows = []
for c in numeric:
    rho, p = stats.spearmanr(data[c], data[TARGET])
    rows.append((c, "numeric", abs(rho), p))
for c in categorical:
    v = cramers_v(data[c], data[TARGET])
    p = stats.chi2_contingency(pd.crosstab(data[c], data[TARGET]))[1]
    rows.append((c, "categorical", v, p))
association = pd.DataFrame(rows, columns=["feature", "type", "effect_size", "p_value"])
association["significant_at_5_percent"] = association["p_value"] < 0.05
association.sort_values("effect_size", ascending=False).round(4).reset_index(drop=True)

,feature,type,effect_size,p_value,significant_at_5_percent
0,EmpEnvironmentSatisfaction,numeric,0.3966,0.0000,True
1,EmpLastSalaryHikePercent,numeric,0.2737,0.0000,True
2,YearsSinceLastPromotion,numeric,0.2697,0.0000,True
3,EmpJobRole,categorical,0.2018,0.0000,True
4,EmpDepartment,categorical,0.1712,0.0000,True
5,ExperienceYearsInCurrentRole,numeric,0.1661,0.0000,True
6,YearsWithCurrManager,numeric,0.1350,0.0000,True
7,ExperienceYearsAtThisCompany,numeric,0.1325,0.0000,True
8,EmpWorkLifeBalance,numeric,0.1137,0.0001,True
9,OverTime,categorical,0.0970,0.0035,True


*Effect size is the absolute Spearman correlation for numeric columns and Cramér's V for categorical columns.*

### 3.2 Model based importance (random forest) and permutation importance

In [7]:
X = data[features]
y = data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), categorical)], remainder="passthrough")
pipe = Pipeline([("prep", prep),
                 ("forest", RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))])
pipe.fit(X_train, y_train)

# impurity importance: add up the one hot columns belonging to each original feature
names = pipe.named_steps["prep"].get_feature_names_out()
raw_importance = pd.Series(pipe.named_steps["forest"].feature_importances_, index=names)
impurity = {}
for name, value in raw_importance.items():
    original = next((c for c in categorical if name.startswith(f"cat__{c}_")), name.replace("remainder__", ""))
    impurity[original] = impurity.get(original, 0) + value
impurity = pd.Series(impurity, name="forest_importance")

perm = permutation_importance(pipe, X_test, y_test, scoring="f1_macro", n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1)
permutation = pd.Series(perm.importances_mean, index=X_test.columns, name="permutation_importance")
print("Held out accuracy of this exploration model:", round(pipe.score(X_test, y_test), 3))
pd.concat([impurity, permutation], axis=1).sort_values("forest_importance", ascending=False).round(4).head(12)

Held out accuracy of this exploration model: 0.917


,forest_importance,permutation_importance
EmpLastSalaryHikePercent,0.1941,0.2461
EmpEnvironmentSatisfaction,0.1815,0.3023
YearsSinceLastPromotion,0.0759,0.1039
EmpJobRole,0.0545,0.0358
EmpDepartment,0.0379,0.0691
ExperienceYearsInCurrentRole,0.0363,0.0550
EmpHourlyRate,0.0360,-0.0017
ExperienceYearsAtThisCompany,0.0329,0.0204
YearsWithCurrManager,0.0320,0.0243
Age,0.0316,0.0065


### 3.3 Combine the three views into one ranking

In [8]:
ranking = association.set_index("feature")[["effect_size"]].join(impurity).join(permutation)
for column in ["effect_size", "forest_importance", "permutation_importance"]:
    ranking[f"rank_{column}"] = ranking[column].rank(ascending=False)
ranking["average_rank"] = ranking[[c for c in ranking.columns if c.startswith("rank_")]].mean(axis=1)
ranking = ranking.sort_values("average_rank").round(4)
ranking.head(10)

,effect_size,forest_importance,permutation_importance,rank_effect_size,rank_forest_importance,rank_permutation_importance,average_rank
feature,,,,,,,
EmpEnvironmentSatisfaction,0.3966,0.1815,0.3023,1.0,2.0,1.0,1.3333
EmpLastSalaryHikePercent,0.2737,0.1941,0.2461,2.0,1.0,2.0,1.6667
YearsSinceLastPromotion,0.2697,0.0759,0.1039,3.0,3.0,3.0,3.0000
EmpDepartment,0.1712,0.0379,0.0691,5.0,5.0,4.0,4.6667
EmpJobRole,0.2018,0.0545,0.0358,4.0,4.0,6.0,4.6667
ExperienceYearsInCurrentRole,0.1661,0.0363,0.0550,6.0,6.0,5.0,5.6667
YearsWithCurrManager,0.1350,0.0320,0.0243,7.0,9.0,7.0,7.6667
ExperienceYearsAtThisCompany,0.1325,0.0329,0.0204,8.0,8.0,8.0,8.0000
EmpWorkLifeBalance,0.1137,0.0279,0.0192,9.0,12.0,10.0,10.3333


In [9]:
top_three = list(ranking.index[:3])
print("Top 3 factors affecting employee performance:", top_three)
PROCESSED.mkdir(parents=True, exist_ok=True)
ranking.reset_index().rename(columns={"index": "feature"}).to_csv(PROCESSED / "feature_ranking.csv", index=False)

Top 3 factors affecting employee performance: ['EmpEnvironmentSatisfaction', 'EmpLastSalaryHikePercent', 'YearsSinceLastPromotion']


**Findings – top three factors**

1. **Environment satisfaction** (strongest on all three views)
2. **Last salary hike percentage**
3. **Years since last promotion**

These three are consistently at the top of the effect size ranking, the random forest importance and the permutation importance, so the choice does not depend on one method.

## 4. Look at the top factors directly

In [10]:
for column in top_three:
    print("=" * 70)
    print(column)
    if data[column].nunique() > 8:
        binned = pd.qcut(data[column], 4, duplicates="drop")
        table = data.groupby(binned)[TARGET].agg(employees="count", average_rating="mean")
        low = data.groupby(binned)[TARGET].apply(lambda s: (s == 2).mean() * 100).rename("percent_rating_2")
        high = data.groupby(binned)[TARGET].apply(lambda s: (s == 4).mean() * 100).rename("percent_rating_4")
        print(table.join(low).join(high).round(2))
    else:
        table = data.groupby(column)[TARGET].agg(employees="count", average_rating="mean")
        low = data.groupby(column)[TARGET].apply(lambda s: (s == 2).mean() * 100).rename("percent_rating_2")
        high = data.groupby(column)[TARGET].apply(lambda s: (s == 4).mean() * 100).rename("percent_rating_4")
        print(table.join(low).join(high).round(2))

EmpEnvironmentSatisfaction
                            employees  average_rating  percent_rating_2  \
EmpEnvironmentSatisfaction                                                
1                                 230            2.67             39.13   
2                                 242            2.65             40.50   
3                                 367            3.14              0.82   
4                                 361            3.13              0.83   

                            percent_rating_4  
EmpEnvironmentSatisfaction                    
1                                       5.65  
2                                       5.79  
3                                      14.71  
4                                      14.13  
EmpLastSalaryHikePercent
                          employees  average_rating  percent_rating_2  \
EmpLastSalaryHikePercent                                                
(10.999, 12.0]                  324            2.83             17.90

**Findings – what the top factors show**

- **Environment satisfaction:** employees with a low or medium score (1 or 2) have about 40 percent rated 2, but only about 0.8 percent of employees with a high or very high score (3 or 4) are rated 2. This is the clearest signal in the data.
- **Salary hike:** the highest hike group (above 18 percent) has a clearly higher average rating (3.30 versus 2.83 to 2.89) because it contains far more rating 4 employees.
- **Promotion gap:** employees promoted within the last year are rated 2 in about 9 percent of cases, compared with about 28 to 30 percent for employees waiting more than one year.

## 5. Profile of employees with the lowest rating (rating 2) versus the rest

In [11]:
data["is_lowest_rated"] = (data[TARGET] == 2).astype(int)
profile = data.groupby("is_lowest_rated")[numeric].mean().T.round(2)
profile.columns = ["all_other_employees", "lowest_rated_employees"]
profile["difference"] = (profile["lowest_rated_employees"] - profile["all_other_employees"]).round(2)
profile.reindex(profile["difference"].abs().sort_values(ascending=False).index).head(10)

,all_other_employees,lowest_rated_employees,difference
EmpHourlyRate,65.55,68.22,2.67
ExperienceYearsAtThisCompany,6.69,9.10,2.41
YearsSinceLastPromotion,1.90,3.70,1.80
ExperienceYearsInCurrentRole,4.00,5.79,1.79
TotalWorkExperienceInYears,11.06,12.75,1.69
YearsWithCurrManager,3.86,5.35,1.49
EmpEnvironmentSatisfaction,2.93,1.58,-1.35
Age,36.75,37.80,1.05
DistanceFromHome,9.04,9.84,0.80
EmpJobLevel,2.02,2.30,0.28


**Findings – profile of the lowest rated employees**

Compared with everyone else, employees rated 2 have a much lower environment satisfaction (average 1.58 versus 2.93), have waited longer since their last promotion (3.7 years versus 1.9 years) and have spent more time with the company, in the current role and with the same manager. Gender, marital status and business travel show no strong pattern.

In [12]:
for column in ["OverTime", "Attrition", "BusinessTravelFrequency", "Gender", "MaritalStatus"]:
    print((pd.crosstab(data[column], data[TARGET], normalize="index") * 100).round(1))
    print()

PerformanceRating     2     3     4
OverTime                           
No                 18.3  70.2  11.5
Yes                11.0  79.0   9.9

PerformanceRating     2     3     4
Attrition                          
No                 15.5  73.4  11.2
Yes                20.2  69.7  10.1

PerformanceRating           2     3     4
BusinessTravelFrequency                  
Non-Travel               15.9  70.5  13.6
Travel_Frequently        16.7  69.4  14.0
Travel_Rarely            16.1  74.1   9.8

PerformanceRating     2     3     4
Gender                             
Female             15.8  73.5  10.7
Male               16.4  72.4  11.2

PerformanceRating     2     3     4
MaritalStatus                      
Divorced           13.8  76.5   9.7
Married            18.2  71.7  10.0
Single             14.8  71.9  13.3



## 6. Correlation between the numeric features (multicollinearity check)

In [13]:
corr = data[numeric + [TARGET]].corr()
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().rename("correlation").reset_index()
pairs.columns = ["feature_one", "feature_two", "correlation"]
pairs.loc[pairs["correlation"].abs() >= 0.5].sort_values("correlation", ascending=False).round(3)

,feature_one,feature_two,correlation
125,EmpJobLevel,TotalWorkExperienceInYears,0.784
281,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,0.764
283,ExperienceYearsAtThisCompany,YearsWithCurrManager,0.759
302,ExperienceYearsInCurrentRole,YearsWithCurrManager,0.729
11,Age,TotalWorkExperienceInYears,0.681
223,TotalWorkExperienceInYears,ExperienceYearsAtThisCompany,0.634
282,ExperienceYearsAtThisCompany,YearsSinceLastPromotion,0.620
301,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,0.541
128,EmpJobLevel,ExperienceYearsAtThisCompany,0.540
6,Age,EmpJobLevel,0.509


**Note on correlated features:** several experience related columns are strongly correlated with each other (for example years at the company, years in the current role and years with the current manager). Tree based models handle this well, and the factor ranking looks at several methods, so the top three are not caused by this overlap.